In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
#read in all words
words = open('names.txt', 'r').read().splitlines()

# create lookup maps
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [3]:
block_size = 3 # context length: how many characters do we take to predict the next one?
X,Y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print("".join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] # crop and append
X=torch.tensor(X)
Y=torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [4]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [5]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [6]:
C = torch.randn((27,2)).float()

In [7]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [16]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)
h = torch.tanh(emb.view(-1,6) @ W1 + b1)
h.shape

torch.Size([32, 100])

In [17]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [18]:
logits = h @ W2 + b2

In [19]:
logits.shape

torch.Size([32, 27])

In [20]:
logits

tensor([[ 4.0990e+00,  4.2791e+00,  9.2775e+00, -1.6679e+00,  5.3506e-01,
          4.5747e+00,  6.5060e+00, -7.6955e+00, -7.0389e-02,  2.6597e+00,
          7.7861e+00, -9.5884e+00,  1.6101e+00,  4.3171e+00,  6.6119e+00,
          1.3131e+01, -4.4072e+00,  2.5894e+00, -4.5490e-01,  5.2405e+00,
          4.5509e+00,  6.7972e+00,  5.8578e+00, -1.4643e+00, -4.5241e-01,
          2.4293e+00,  6.9637e-02],
        [ 1.1658e+01, -3.6814e+00,  2.6912e+00,  9.1944e-01,  9.4660e+00,
         -1.5704e+00,  3.0365e+00, -4.5295e+00, -6.0951e-01,  1.7453e+01,
          3.6425e+00, -5.6365e+00,  3.5507e+00,  5.2083e+00,  5.4165e+00,
          1.1439e+00, -1.2896e+00,  7.0337e+00, -2.2601e+00,  7.2124e+00,
          3.4136e+00, -6.7125e+00,  2.1709e+01, -1.5961e+00, -7.7650e+00,
          8.5761e-01,  6.1799e+00],
        [-6.0883e+00, -6.9015e+00,  1.2323e+01, -3.3240e-01,  8.9572e+00,
          1.1941e+01,  6.8029e+00, -5.1165e+00, -3.3821e+00, -4.2432e+00,
          6.4299e+00,  5.0343e+00, -2.08

In [21]:
counts = logits.exp()

In [22]:
prob = counts/counts.sum(1, keepdims=True)

In [23]:
prob.shape

torch.Size([32, 27])

In [25]:
prob[0].sum()

tensor(1.)

In [ ]:
Y